In [1]:
import statistics

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def prepare_data() -> TensorDataset:
    X = torch.randn(10000, 128)
    y = torch.randint(0, 2, (10000,))
    return TensorDataset(X, y)

In [3]:
def train():
    # pin_memory + workers чтобы подгрузка батчей не блокировала основной поток
    dataloader = DataLoader(
        prepare_data(),
        batch_size=256,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )

    model = nn.Sequential(
        nn.Linear(128, 512), nn.ReLU(),
        nn.Linear(512, 128), nn.ReLU(),
        nn.Linear(128, 2),
    ).cuda().train()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    losses_history = []
    forward_events = []
    backward_events = []

    for batch_idx, (data, target) in enumerate(dataloader):
        # non_blocking работает только с pin_memory, копия идёт параллельно с вычислениями
        data = data.to('cuda', non_blocking=True)
        target = target.to('cuda', non_blocking=True)
        # noise сразу на gpu, иначе лишняя копия с cpu на каждом шаге
        data = data + torch.randn_like(data)

        optimizer.zero_grad(set_to_none=True)

        # time.time вокруг cuda мерит только постановку в очередь, а не само выполнение.
        # поэтому меряем через cuda events
        fwd_s = torch.cuda.Event(enable_timing=True)
        fwd_e = torch.cuda.Event(enable_timing=True)
        bwd_s = torch.cuda.Event(enable_timing=True)
        bwd_e = torch.cuda.Event(enable_timing=True)

        fwd_s.record()
        output = model(data)
        loss = criterion(output, target)
        fwd_e.record()

        bwd_s.record()
        loss.backward()
        bwd_e.record()

        optimizer.step()

        # без detach в список попадает весь граф вычислений и память течёт
        losses_history.append(loss.detach())
        forward_events.append((fwd_s, fwd_e))
        backward_events.append((bwd_s, bwd_e))

    # один sync в конце, а не на каждом шаге, чтобы не ломать асинхронность
    torch.cuda.synchronize()

    forward_times = [s.elapsed_time(e) / 1000 for s, e in forward_events]
    backward_times = [s.elapsed_time(e) / 1000 for s, e in backward_events]
    losses_history = [l.item() for l in losses_history]

    for i, l in enumerate(losses_history):
        print(f"Batch {i} loss: {l:.4f}")

    print(
        f"Epoch finished, avg forward time is {statistics.mean(forward_times)}, "
        f"avg backward time is {statistics.mean(backward_times)}"
    )


# empty_cache из цикла убрал, он не чинит утечки, только тормозит

In [4]:
train()

Batch 0 loss: 0.6948
Batch 1 loss: 0.6910
Batch 2 loss: 0.7025
Batch 3 loss: 0.7006
Batch 4 loss: 0.6913
Batch 5 loss: 0.6944
Batch 6 loss: 0.7004
Batch 7 loss: 0.6961
Batch 8 loss: 0.7003
Batch 9 loss: 0.7024
Batch 10 loss: 0.6969
Batch 11 loss: 0.7139
Batch 12 loss: 0.6964
Batch 13 loss: 0.6965
Batch 14 loss: 0.7049
Batch 15 loss: 0.7138
Batch 16 loss: 0.7063
Batch 17 loss: 0.6922
Batch 18 loss: 0.6950
Batch 19 loss: 0.6900
Batch 20 loss: 0.6947
Batch 21 loss: 0.6866
Batch 22 loss: 0.6918
Batch 23 loss: 0.6927
Batch 24 loss: 0.6980
Batch 25 loss: 0.7026
Batch 26 loss: 0.6922
Batch 27 loss: 0.6934
Batch 28 loss: 0.6963
Batch 29 loss: 0.6893
Batch 30 loss: 0.6974
Batch 31 loss: 0.6891
Batch 32 loss: 0.6938
Batch 33 loss: 0.6966
Batch 34 loss: 0.6979
Batch 35 loss: 0.6860
Batch 36 loss: 0.6949
Batch 37 loss: 0.6951
Batch 38 loss: 0.6862
Batch 39 loss: 0.6630
Epoch finished, avg forward time is 0.008321031686663627, avg backward time is 0.005382528720796109
